In [1]:
from dotenv import load_dotenv

In [2]:
_env = '../.env'

In [3]:
load_dotenv(_env)

True

In [4]:
import os
openai_key = os.environ.get('OPENAI_API_KEY')

In [5]:
from openai import OpenAI

In [6]:
client = OpenAI(api_key=openai_key)

## Simple API call

In [7]:
SYSTEM_PROMPT = """
You are a helpful personal assistant. You will help answer questions the user has.
"""

In [19]:
resp = client.chat.completions.create(
    model='gpt-5.4-mini',
    temperature = 1,
    messages=[
        {'role':'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'Hello! Who are you?'}
    ],
    n=3
)

In [20]:
for i in range(3):
    print('i=',i)
    print(resp.choices[i].message.content)


i= 0
Hello! I’m an AI assistant here to help answer questions, explain things, and assist with a wide range of tasks. How can I help you today?
i= 1
Hello! I’m an AI assistant here to help answer questions, explain things, and assist with a wide range of tasks. How can I help you today?
i= 2
Hello! I’m an AI assistant here to help answer questions, brainstorm, write, explain things, and assist with tasks. How can I help you today?


In [21]:
resp.choices[0].message.content

'Hello! I’m an AI assistant here to help answer questions, explain things, and assist with a wide range of tasks. How can I help you today?'

In [22]:
resp.choices[0].finish_reason

'stop'

## Tool calls

In [23]:
# ── Tool definitions and mock implementations ─────────────────────────────

# OpenAI tool schema format
oai_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city. Call this when the user asks about weather.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. 'Paris'"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"],
                             "description": "Temperature unit. Defaults to celsius."},
                },
                "required": ["city"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression and return the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string",
                                   "description": "Math expression string, e.g. '2 + 3 * 4'"},
                },
                "required": ["expression"],
                "additionalProperties": False,
            },
        },
    },
]

# Mock implementations
def get_weather(city: str, unit: str = "celsius") -> dict:
    db = {"london": 14, "paris": 19, "new york": 23, "tokyo": 26, "sydney": 17}
    temp = db.get(city.lower(), 20)
    if unit == "fahrenheit":
        temp = round(temp * 9 / 5 + 32, 1)
    return {"city": city, "temperature": temp, "unit": unit, "condition": "partly cloudy"}

def calculate(expression: str) -> dict:
    # NEVER use eval on untrusted user input in production!
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"error": str(e)}

def execute_tool(name: str, args: dict) -> str:
    """Dispatch tool call to implementation, return JSON string."""
    if name == "get_weather":
        return json.dumps(get_weather(**args))
    elif name == "calculate":
        return json.dumps(calculate(**args))
    return json.dumps({"error": f"Unknown tool: {name}"})

print("Tools defined:", [t["function"]["name"] for t in oai_tools])
print("Test get_weather:", get_weather("Paris"))
print("Test calculate:", calculate("15 * 23 + 7"))

Tools defined: ['get_weather', 'calculate']
Test get_weather: {'city': 'Paris', 'temperature': 19, 'unit': 'celsius', 'condition': 'partly cloudy'}
Test calculate: {'expression': '15 * 23 + 7', 'result': 352}


In [58]:
resp = client.chat.completions.create(
    model='gpt-5.4-mini',
    temperature=0,
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'what is the wehther in Paris?'}
    ],
    tools = oai_tools,
    tool_choice='auto'
)

In [59]:
resp.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_x8EN91dOYqvXEQ9gl94gfiHd', function=Function(arguments='{"city":"Paris","unit":"celsius"}', name='get_weather'), type='function')])

In [60]:
resp.choices[0].finish_reason

'tool_calls'

In [30]:
resp.choices[0].message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_M5mIQ9KCgKcFQbCz5aDjSa4s', function=Function(arguments='{"city":"Paris","unit":"celsius"}', name='get_weather'), type='function')]

In [32]:
resp.choices[0].message.tool_calls[0].function

Function(arguments='{"city":"Paris","unit":"celsius"}', name='get_weather')

In [33]:
tc = resp.choices[0].message.tool_calls[0]

In [37]:
import json
args = json.loads(tc.function.arguments)

In [39]:
result = execute_tool(tc.function.name, args)

In [41]:
result

'{"city": "Paris", "temperature": 19, "unit": "celsius", "condition": "partly cloudy"}'

In [43]:
messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'what is the wehther in Paris?'}
    ]

In [44]:
messages.append(resp.choices[0].message)

In [51]:
resp.choices[0].message.tool_calls[0].id

'call_M5mIQ9KCgKcFQbCz5aDjSa4s'

In [45]:
messages

[{'role': 'system',
  'content': '\nYou are a helpful personal assistant. You will help answer questions the user has.\n'},
 {'role': 'user', 'content': 'what is the wehther in Paris?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_M5mIQ9KCgKcFQbCz5aDjSa4s', function=Function(arguments='{"city":"Paris","unit":"celsius"}', name='get_weather'), type='function')])]

In [52]:
messages.append(
    {
        'role': 'tool',
        'tool_call_id': resp.choices[0].message.tool_calls[0].id,
        'content': result
    }
)

In [53]:
messages

[{'role': 'system',
  'content': '\nYou are a helpful personal assistant. You will help answer questions the user has.\n'},
 {'role': 'user', 'content': 'what is the wehther in Paris?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_M5mIQ9KCgKcFQbCz5aDjSa4s', function=Function(arguments='{"city":"Paris","unit":"celsius"}', name='get_weather'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_M5mIQ9KCgKcFQbCz5aDjSa4s',
  'content': '{"city": "Paris", "temperature": 19, "unit": "celsius", "condition": "partly cloudy"}'}]

In [54]:
resp = client.chat.completions.create(
    model='gpt-5.4-mini',
    temperature=0,
    messages= messages
)

In [56]:
resp.choices[0].message.content

'The weather in Paris is **19°C** and **partly cloudy**.'

In [63]:
user_query = "What's the weather in Paris and London, and what's 15 * 23?"

In [68]:
messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
messages.append({'role': 'user', 'content': user_query})

max_iters = 5
for _ in range(max_iters):
    resp = client.chat.completions.create(
        model='gpt-5.4-mini',
        temperature=0,
        messages=messages,
        tools=oai_tools,
        tool_choice='auto'
    )
    msg = resp.choices[0].message
    messages.append(msg)

    if resp.choices[0].finish_reason == 'tool_calls':
        for tool_call in msg.tool_calls:
            func = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            id = tool_call.id
            result = execute_tool(func, args)
            messages.append(
                {
                    'role': 'tool',
                    'tool_call_id': id,
                    'content': result
                }
            )
    else:
        print(msg.content)
        break
    

Paris: 19°C, partly cloudy  
London: 14°C, partly cloudy  
15 × 23 = 345


## Multi-modal

In [69]:
import base64

In [73]:
with open('./assets/sample_multimodal.png', 'rb') as f:
    IMG_B64 = base64.standard_b64encode(f.read()).decode("utf-8")

In [74]:
resp = client.chat.completions.create(
    model='gpt-5.4-mini',
    temperature=0,
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': [
            {'type': 'text', 'text': 'Can you describe the picture for me?'},
            {'type': 'image_url',
             'image_url': {
                 'url': f'data:image/png;base64,{IMG_B64}'
             }}
        ]}
    ]
)

In [76]:
print(resp.choices[0].message.content)

The picture shows four colorful, semi-transparent dice floating against a bright, abstract background.

- A **blue die** is near the upper left.
- A **green die** is near the upper right.
- A **red die** is in the center and appears to be the largest and most prominent.
- A **yellow die** is partly visible near the bottom, tucked behind the red one.

All of the dice have **white pips** and a glossy, glass-like look. The background is made of bold horizontal bands of color, including blue, green, red, yellow, and black, giving the image a vivid, stylized feel.


## React agent

In [77]:
oai_tools[0]

{'type': 'function',
 'function': {'name': 'get_weather',
  'description': 'Get current weather for a city. Call this when the user asks about weather.',
  'parameters': {'type': 'object',
   'properties': {'city': {'type': 'string',
     'description': "City name, e.g. 'Paris'"},
    'unit': {'type': 'string',
     'enum': ['celsius', 'fahrenheit'],
     'description': 'Temperature unit. Defaults to celsius.'}},
   'required': ['city'],
   'additionalProperties': False}}}

In [79]:
new_tool = [
    {
        'type': 'function',
        'function': {
            'name': 'search_web',
            'description': 'Search the web for current information on a topic.',
            'parameters':{
                'type': 'object',
                'properties': {
                    'query': {
                        'type': 'string',
                        'description': 'Search query',
                    },
                },
                'required': ['query'],
                'additionalProperties': False,
            }
        }
    }
]

In [80]:
oai_tools += new_tool

In [81]:
len(oai_tools)

3

In [82]:
def search_web(query: str) -> dict:
    return {"results": [f"Mocked search result for: {query}",
                        "LLMs have context windows measured in tokens."]}

In [83]:
def execute_tool(name: str, args: dict) -> str:
    if name == "get_weather":
        return json.dumps(get_weather(**args))
    elif name == "calculate":
        return json.dumps(calculate(**args))
    elif name == "search_web":
        return json.dumps(search_web(**args))
    return json.dumps({"error": f"Unknown tool: {name}"})

In [84]:
REACT_SYSTEM = """You are a helpful assistant with access to tools.
Think step-by-step before acting. Use tools when you need external information.
After getting tool results, synthesize them into a clear final answer."""

In [96]:
query = "What's the combined temperature of Paris and Tokyo in Celsius, and what is that sum divided by 2?"

In [104]:
def react_agent(query, max_iters= 8):
    messages = [
        {'role': 'system', 'content': REACT_SYSTEM},
        {'role': 'user', 'content': query}
    ]
    
    for _ in range(max_iters):
    
        resp = client.chat.completions.create(
            model='gpt-5.4-mini',
            temperature=0,
            messages=messages,
            tools=oai_tools,
            tool_choice = 'auto'
        )
        msg = resp.choices[0].message
        messages.append(msg)
    
        if resp.choices[0].finish_reason == 'tool_calls':
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                func = tc.function.name
                id = tc.id
                result = execute_tool(func, args)
                messages.append({
                    'role': 'tool',
                    'tool_call_id': id,
                    'content': result
                })
        else:
            return msg.content
    
    return 'Max iteration reached!'

In [105]:
answer = react_agent(query)

In [107]:
print(answer)

Paris is 19°C and Tokyo is 26°C.

- Combined temperature: **45°C**
- Divided by 2: **22.5°C**


## Schema and LLM-as-judge

In [124]:
judge_output_schema = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'judge_output_format',
        'schema': {
            'type': 'object',
            'properties': {
                'accuracy': {
                    'type': 'number',
                    'minimum': 0,
                    'maximum': 1,
                    'description': 'Factual correctness of the answer',
                },
                'completeness': {
                    'type': 'number',
                    'minimum': 0,
                    'maximum': 1,
                    'description': 'Does it fully address the question?',
                },
                'clarity': {
                    'type': 'number',
                    'minimum': 0,
                    'maximum': 1,
                    'description': 'Is the answer clear and well-writen?',
                },
                'issues': {
                    'type': 'array',
                    'items': {'type': 'string'},
                    'description': 'List of sepecific problems with the answer',
                },
                'overall_feedback': {'type':'string'},
            },
            'required': ['accuracy', 'completeness', 'clarity', 'overall_feedback'],
            'additionalProperties': False
        }
    }
    
}

In [125]:
JUDGE_SYSTEM = """
You are an expert evaluator assessing LLM-generated answers.
Score each criterion 0-1 where 1 is perfect. Be critical and specific about issues.
"""

In [126]:
def llm_judge(question, answer, schema):
    resp = client.chat.completions.create(
        model='gpt-5.4-mini',
        temperature=0,
        messages=[
            {'role': 'system', 'content': JUDGE_SYSTEM},
            {'role': 'user', 'content': f"Question: {question} \n\n Answer to evaluate: {answer}"}
        ],
        response_format = schema
    )
    return resp.choices[0].message.content

In [127]:
question = 'What is gradient descent?'
answer = 'Gradient descent is an iterative optimization algorithm that minimizes a loss function by updating parameters in the direction opposite to the gradient, scaled by the learning rate. It is the backbone of training neural networks.'

In [128]:
result = llm_judge(question, answer, judge_output_schema)

In [130]:
json.loads(result)

{'accuracy': 1,
 'completeness': 0.9,
 'clarity': 1,
 'issues': ['The answer is correct but slightly brief; it does not mention that gradient descent is used more broadly than neural networks.',
  'It omits a simple intuition/example of how the updates work, though this is not strictly necessary.'],
 'overall_feedback': 'The response gives a correct and clear definition of gradient descent and appropriately mentions its role in neural network training. It is slightly incomplete because it focuses on one application and does not add intuition or broader context, but overall it answers the question well.'}